# Reclassement (Reranking) Pinecone et Recherche Vectorielle

## Partie 1 : Charger les documents et exécuter le modèle de reclassement

### 1. Installer les bibliothèques Pinecone

**Que faire** : Exécutez cette commande dans la cellule de votre carnet (notez le `!` pour l'exécution dans le carnet).
**Pourquoi** : Vous aurez besoin du package client pour interagir avec l'API de Pinecone et de l'assistant notebook pour simplifier l'authentification dans des environnements comme Colab.
**💡 Indice** : Si vous avez des conflits de version, redémarrez votre temps d'exécution après l'installation.

In [5]:
# On installe le client Python de Pinecone (version 6.0.1)
# et le package "pinecone-notebooks" qui facilite l'authentification dans Colab
!pip install -U pinecone==6.0.1 pinecone-notebooks

### 2. S'authentifier avec Pinecone

**Que faire** : Exécutez ce bloc de code exactement comme indiqué. Il demandera votre clé API si elle n'est pas déjà configurée.
**Pourquoi** : Fournir de manière sécurisée votre clé API permet au client de se connecter à votre projet Pinecone sans coder en dur les secrets dans votre script.
**Indice** : Récupérez votre clé API depuis votre tableau de bord Pinecone, dans la section « API Keys ». Gardez-la secrète !

In [6]:
import os
from pinecone import Pinecone
from pinecone_notebooks.colab import Authenticate

# On regarde si la clé API est déjà présente dans les variables d'environnement
api_key = os.environ.get("PINECONE_API_KEY")

if not api_key:
    # Si la clé n'existe pas encore, on lance la fenêtre d'authentification Pinecone
    print("La clé API Pinecone n'est pas définie. Merci de la fournir dans la fenêtre qui va s'ouvrir.")
    Authenticate()  # Cette fonction demande la clé et la stocke dans os.environ["PINECONE_API_KEY"]
    api_key = os.environ.get("PINECONE_API_KEY")  # On récupère la clé une nouvelle fois

# Une fois la clé récupérée, on peut créer le client Pinecone
if api_key:
    pc = Pinecone(api_key=api_key)
    print("Client Pinecone initialisé avec succès.")
else:
    print("ERREUR : impossible d'initialiser le client Pinecone. La clé API n'a pas été fournie.")

Client Pinecone initialisé avec succès.


### 3. Instancier le client Pinecone

**Que faire** : Rien de plus à faire ici : le client `pc` a déjà été créé dans la cellule précédente, qui gère l'authentification et l'instanciation en une seule étape.
**Pourquoi** : Le client (`pc`) est votre point d'entrée pour toutes les opérations Pinecone — création d'index, requêtes et reclassement.
**💡 Indice** : Si vous recevez des erreurs d'authentification, assurez-vous que votre clé API est correcte et active.

In [7]:
# Cette cellule ne fait rien de plus : le client "pc" a déjà été créé juste au-dessus,
# lors de l'étape d'authentification. On garde cette cellule pour respecter la structure du TP.
print("Client Pinecone déjà prêt :", pc)

Client Pinecone déjà prêt : <pinecone.control.pinecone.Pinecone object at 0x7bfe084a2390>


### 4. Définir la requête et les documents

**Que faire** : Remplacez chaque `...` par un vrai texte de document, en mélangeant des références à Apple (l'entreprise) et à apple (le fruit).
**Pourquoi** : Vous avez besoin d'un petit jeu de documents pour tester la capacité du reranker à distinguer les différents contextes d'un même mot.
**💡 Indice** : Faites certains documents sur « Apple, le fruit » et d'autres sur « Apple, l'entreprise qui fabrique des iPhones » — cela teste la compréhension contextuelle.

In [8]:
# La requête que l'on va utiliser pour tester le reclassement
query = "Tell me about Apple's products"

# On crée un petit jeu de documents qui mélange volontairement :
# - des documents parlant du fruit "apple"
# - des documents parlant de l'entreprise "Apple"
documents = [
    "The apple is a deciduous tree in the rose family best known for its sweet, pomaceous fruit, the apple.",  # à propos du fruit
    "Apple Inc. is an American multinational technology company headquartered in Cupertino, California, that designs, develops, and sells consumer electronics, computer software, and online services.",  # à propos de l'entreprise
    "An apple a day keeps the doctor away.",  # encore à propos du fruit
    "Apple announced its new iPhone model at the recent keynote event.",  # encore à propos de l'entreprise
    "Granny Smith is a popular type of green apple."  # un dernier document au choix (fruit)
]

### 5. Appeler le reranker

**Que faire** : Remplissez `top_n` avec le nombre de meilleurs résultats que vous voulez récupérer (par exemple 3).
**Pourquoi** : `top_n` limite le nombre de résultats reclassés, afin de ne récupérer que les documents les plus pertinents.
**💡 Indice** : Essayez différentes valeurs de `top_n` pour voir comment le classement change !

In [9]:
from pinecone import RerankModel

# On envoie nos documents au modèle de reclassement "bge-reranker-v2-m3"
# Ce modèle va comparer chaque document à la requête et leur donner un score de pertinence
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3  # on ne garde que les 3 meilleurs résultats
)

### 6. Inspecter les résultats reclassés

**Que faire** : Écrivez le code qui affiche le rang (`i+1`), le score de similarité (`m.score`) et le texte du document (`m.document.text`). Trouvez aussi le bon attribut sur `reranked` pour accéder à la liste des résultats.
**Pourquoi** : Voir ces valeurs montre comment le reranker ordonne les documents et quels scores il leur attribue.
**💡 Indice** : Regardez la structure de l'objet `reranked`. L'attribut qui contient la liste des résultats s'appelle `.data` (et non `.results`). Un score plus élevé = plus pertinent !

In [10]:
def show_reranked_results(query, matches):
    # On affiche la requête d'origine
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        # Pour chaque résultat : son rang (i+1), son score de pertinence, et le texte du document
        print(f"{str(i+1).rjust(4)}. Score: {m.score:.4f}, Document: {m.document.text}")

# ATTENTION : l'attribut correct est ".data" et non ".results"
show_reranked_results(query, reranked.data)

Query: Tell me about Apple's products
   1. Score: 0.5838, Document: Apple Inc. is an American multinational technology company headquartered in Cupertino, California, that designs, develops, and sells consumer electronics, computer software, and online services.
   2. Score: 0.0170, Document: Apple announced its new iPhone model at the recent keynote event.
   3. Score: 0.0162, Document: The apple is a deciduous tree in the rose family best known for its sweet, pomaceous fruit, the apple.


## Partie 2 : Mettre en place un index serverless pour des notes médicales

### 1. Installer les bibliothèques de données et de modèle

**Que faire** : Exécutez cette commande d'installation dans une cellule du carnet.
**Pourquoi** : Vous utiliserez ces bibliothèques pour charger, transformer en vecteurs (embeddings) et manipuler les données de notes médicales.
**💡 Indice** : Cela peut prendre quelques minutes. Profitez-en pour aller chercher un café !

In [11]:
# pandas : pour manipuler les données tabulaires
# torch : pour faire tourner le modèle d'embedding
# transformers : pour charger le modèle de langage pré-entraîné
!pip install pandas torch transformers

### 2. Importer les modules et définir les paramètres d'environnement

**Que faire** : Renseignez le fournisseur cloud (par exemple 'aws'), la région (par exemple 'us-east-1'), et choisissez un nom pour votre index.
**Pourquoi** : Vous configurez ici un index serverless adapté à vos besoins, dans la bonne région cloud.
**💡 Indice** : La plupart des comptes Pinecone utilisent 'aws' et 'us-east-1'. Pour le nom de l'index, essayez quelque chose comme `medical-notes-index`.

In [12]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# On récupère le fournisseur cloud et la région (valeurs par défaut qui conviennent à la plupart des comptes)
cloud = os.getenv('PINECONE_CLOUD', 'aws')       # ex: 'aws'
region = os.getenv('PINECONE_REGION', 'us-east-1')  # ex: 'us-east-1'

# On définit les spécifications de l'index serverless
spec = ServerlessSpec(cloud=cloud, region=region)

# On choisit un nom pour notre index
index_name = 'medical-notes-index'

### 3. Créer (ou recréer) l'index

**Que faire** : Renseignez la dimension (384 pour notre modèle) et choisissez une métrique de distance ('cosine' est recommandé).
**Pourquoi** : La dimension de l'index doit correspondre à celle des vecteurs d'embedding que vous allez insérer, sinon l'upsert échouera.
**💡 Indice** : Le modèle d'embedding que nous utilisons produit des vecteurs de 384 dimensions. La similarité cosinus fonctionne bien pour du texte.

In [13]:
# On supprime d'abord un éventuel index existant portant le même nom (pour repartir de zéro)
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# On crée un nouvel index
pc.create_index(
    name=index_name,
    dimension=384,     # doit correspondre à la taille des vecteurs produits par notre modèle d'embedding
    metric='cosine',   # la similarité cosinus est adaptée pour comparer des textes
    spec=spec
)

{
    "name": "medical-notes-index",
    "metric": "cosine",
    "host": "medical-notes-index-u7rtm20.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

## Partie 3 : Charger les données d'exemple

### 1. Télécharger et lire le fichier JSONL

**Que faire** : Insérez l'URL brute ("raw") du fichier GitHub contenant les notes médicales.
**Pourquoi** : Cela télécharge des exemples de notes médicales déjà prêtes à être utilisées.
**💡 Indice** : Utilisez bien l'URL « raw » de GitHub, et non l'URL de visualisation classique du fichier.

In [14]:
import requests
import tempfile
import os
import pandas as pd

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # URL brute ("raw") du fichier de données sur GitHub
    # Remarque : c'est l'URL qui fonctionne réellement (sans "refs/heads/", qui renvoie une erreur 404)
    url = "https://raw.githubusercontent.com/pinecone-io/examples/master/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()  # on vérifie que le téléchargement a bien fonctionné

    # On sauvegarde le fichier téléchargé dans un dossier temporaire
    with open(file_path, "wb") as f:
        f.write(response.content)

    # On charge le fichier JSONL dans un DataFrame pandas
    df = pd.read_json(file_path, orient='records', lines=True)

### 2. Prévisualiser le DataFrame

**Que faire** : Trouvez l'attribut pandas qui affiche les dimensions du DataFrame.
**Pourquoi** : Cela permet de vérifier que vous avez les bonnes colonnes (par exemple `id`, `values`, `metadata`) avant l'upsert.
**💡 Indice** : Quel attribut pandas affiche (nombre de lignes, nombre de colonnes) ?

In [15]:
# .shape donne un tuple (nombre de lignes, nombre de colonnes)
print("Data shape:", df.shape)

# On affiche les premières lignes pour vérifier le contenu
df.head()

Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


## Partie 4 : Insérer les données dans l'index (upsert)

### 1. Instancier le client de l'index et faire l'upsert

**Que faire** : Passez la variable DataFrame à la fonction d'upsert.
**Pourquoi** : Cela envoie tous les embeddings et métadonnées de vos notes vers Pinecone, pour pouvoir les interroger ensuite.
**💡 Indice** : Dans quelle variable avez-vous stocké les données de notes médicales ?

In [16]:
# On instancie un client pour interagir avec notre index
index = pc.Index(name=index_name)

# On insère (upsert) les données du DataFrame directement dans l'index
index.upsert_from_dataframe(df)

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

{'upserted_count': 100}

### 2. Attendre que les données soient disponibles

**Que faire** : Complétez la condition — le nombre de vecteurs doit être supérieur à quelle valeur ?
**Pourquoi** : Cela garantit que les vecteurs insérés sont bien indexés avant de lancer une requête.
**💡 Indice** : On veut attendre qu'il y ait au moins quelques vecteurs dans l'index. Quel est le minimum logique ?

In [17]:
def is_fresh(index):
    # On récupère les statistiques de l'index (dont le nombre total de vecteurs)
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print("Vector count:", vector_count)
    # On considère l'index "prêt" dès qu'il contient au moins 1 vecteur
    return vector_count > 0

# On attend, en vérifiant toutes les 5 secondes, que l'index contienne des données
while not is_fresh(index):
    time.sleep(5)

print("Index prêt !")
index.describe_index_stats()

Vector count: 100
Index prêt !


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

## Partie 5 : Requête et fonction d'embedding

### 1. Définir votre fonction d'embedding

**Que faire** : Indiquez sur quelle dimension faire la moyenne (0 ou 1).
**Pourquoi** : Cela convertit les questions entrantes dans le même espace vectoriel que vos notes indexées.
**💡 Indice** : On veut faire la moyenne sur la dimension de longueur de séquence, afin d'obtenir un seul vecteur par texte en entrée.

In [18]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # On transforme le texte en tokens compréhensibles par le modèle
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')

    # On calcule les embeddings sans suivre les gradients (on ne fait pas d'entraînement)
    with torch.no_grad():
        model_output = model(**encoded_input)

    # last_hidden_state a la forme (batch, longueur_de_sequence, taille_du_vecteur)
    # [0] retire la dimension "batch" -> il reste (longueur_de_sequence, taille_du_vecteur)
    # mean(dim=0) fait donc la moyenne sur la longueur de séquence, pour obtenir UN SEUL vecteur
    embedding = model_output.last_hidden_state[0].mean(dim=0)
    return embedding

### 2. Lancer une requête de recherche sémantique

**Que faire** : Écrivez une question médicale et choisissez le nombre de résultats à récupérer.
**Pourquoi** : Cela récupère les notes les plus proches sémantiquement de votre question, à partir de l'index.
**💡 Indice** : Essayez des questions comme « patient has chest pain » ou « broken bone treatment ». Récupérez 5 à 10 résultats.

In [19]:
# On pose une question médicale
question = "patient has chest pain"

# On transforme la question en vecteur d'embedding, dans le même espace que les notes indexées
query_embedding = get_embedding(question).tolist()

# On interroge l'index pour récupérer les notes les plus proches (top_k = nombre de résultats)
results = index.query(vector=[query_embedding], top_k=5, include_metadata=True)

# On trie les résultats par score décroissant (du plus pertinent au moins pertinent)
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Partie 6 : Afficher et reclasser les notes cliniques

### 1. Afficher les résultats de recherche initiaux

**Que faire** : Trouvez les bonnes clés du dictionnaire pour le score et les métadonnées.
**Pourquoi** : Cela permet de voir quelles notes ont été jugées les plus pertinentes au départ.
**💡 Indice** : Regardez la structure des objets `match` renvoyés par une requête Pinecone.

In [20]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f'      Score: {match["score"]}')       # la clé "score" contient le score de similarité
        print(f'      Metadata: {match["metadata"]}')  # la clé "metadata" contient les infos associées
        print('')

show_results(question, sorted_matches)

Question: 'patient has chest pain'

Results:
   1. ID: P001
      Score: 0.734459221
      Metadata: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

   2. ID: P016
      Score: 0.483537316
      Metadata: {'condition': 'heart murmur', 'referral': 'cardiology'}

   3. ID: P0100
      Score: 0.446578264
      Metadata: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}

   4. ID: P047
      Score: 0.416666359
      Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

   5. ID: P095
      Score: 0.416666359
      Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}



### 2. Préparer les documents pour le reclassement

**Que faire** : Trouvez la bonne clé pour accéder aux métadonnées de chaque résultat.
**Pourquoi** : Cela construit un champ résumant les métadonnées de chaque note, pour que le reranker puisse s'en servir.
**💡 Indice** : Quelle clé avez-vous utilisée à l'étape précédente pour afficher les métadonnées ?

In [21]:
# On crée une liste de documents où chaque document contient :
# - son id
# - un champ "reranking_field" qui résume toutes ses métadonnées sous forme de texte
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

### 3. Exécuter le reclassement serverless

**Que faire** : Créez une question médicale plus précise et choisissez combien de résultats reclassés récupérer.
**Pourquoi** : Le reclassement utilise la question affinée et le champ de métadonnées pour réordonner les notes selon leur nouvelle pertinence.
**💡 Indice** : Essayez « patient needs knee surgery » ou « diabetes treatment plan ». Récupérez 2 à 3 meilleurs résultats.

In [22]:
# On définit une question plus précise pour le reclassement
refined_query = "patient needs knee surgery"

# On effectue le reclassement à partir de cette question et du champ "reranking_field"
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,                 # on ne garde que les 3 meilleurs résultats
    return_documents=True,
)

### 4. Afficher les résultats reclassés

**Que faire** : Trouvez les bons attributs pour le score de reclassement, le champ recherchable, et la collection de résultats.
**Pourquoi** : Cela permet de comparer comment le reranker améliore l'ordre des résultats par rapport à la recherche initiale.
**💡 Indice** : Regardez la structure de l'objet `reranked_results` — cherchez `.score`, le nom des champs, et `.data` (et non `.results`).

In [23]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f'      Score: {match.score}')                                 # le score de reclassement
        print(f'      Reranking Field: {match.document.reranking_field}')    # le champ utilisé pour le reclassement
        print('')

# ATTENTION : l'attribut correct est ".data" et non ".results"
show_reranked_results(refined_query, reranked_results.data)

Question: 'patient needs knee surgery'

Reranked Results:
   1. ID: P047
      Score: 0.013428255
      Reranking Field: symptoms: back pain; treatment: physical therapy

   2. ID: P095
      Score: 0.013428255
      Reranking Field: symptoms: back pain; treatment: physical therapy

   3. ID: P0100
      Score: 0.0013831174
      Reranking Field: advice: over-the-counter pain relief, stretching; symptoms: muscle pain



### 5. Nettoyage (optionnel)

**Que faire** : Exécutez cette cellule une fois que vous avez terminé, pour éviter des coûts inutiles.
**Pourquoi** : Les index serverless coûtent de l'argent tant qu'ils contiennent des données, il faut donc nettoyer après vos expérimentations.
**💡 Indice** : Vous pourrez toujours recréer l'index plus tard si besoin !

In [24]:
# On supprime l'index pour ne pas payer inutilement
pc.delete_index(name=index_name)